#### 0. Configuracion Inicial

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn

df = pd.read_csv('docs/product_activity.csv') # 

#### 1. - Exploracion Inicial

##### 1.1 - Vista general del DataSet

In [11]:
print("Head")
display(df.head())
print("\nInfo")
df.info()
print("\nDescribe")
df.describe()

Head


,user_id,created_at,country,plan_type,user_age,post_id,post_category,post_created_at,votes_received,user_total_posts,days_since_signup,device_type
0,U01988,2025-02-18T02:07:44,PY,pro,26.0,P0008515,sport,2025-05-07T20:55:28,7,16,78,mobile
1,U00236,2025-06-22T07:49:10,BR,free,27.0,P0001023,tech,2025-09-13T20:31:06,1,9,83,web
2,U00791,2024-02-12T02:45:45,CL,free,28.0,P0003405,tech,2024-02-14T05:17:48,11,2,2,mobile
3,U01522,2024-09-22T07:06:50,US,free,16.0,P0006524,finance,2024-09-24T07:51:34,5,2,2,web
4,U01092,2025-07-18T02:27:52,PY,free,NaN,P0004665,education,2025-07-24T04:56:56,7,2,6,mobile



Info
<class 'pandas.DataFrame'>
RangeIndex: 8782 entries, 0 to 8781
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            8782 non-null   str    
 1   created_at         8782 non-null   str    
 2   country            8782 non-null   str    
 3   plan_type          8782 non-null   str    
 4   user_age           8028 non-null   float64
 5   post_id            8782 non-null   str    
 6   post_category      8782 non-null   str    
 7   post_created_at    8782 non-null   str    
 8   votes_received     8782 non-null   int64  
 9   user_total_posts   8782 non-null   int64  
 10  days_since_signup  8782 non-null   int64  
 11  device_type        8782 non-null   str    
dtypes: float64(1), int64(3), str(8)
memory usage: 823.4 KB

Describe


,user_age,votes_received,user_total_posts,days_since_signup
count,8028.000000,8782.000000,8782.000000,8782.000000
mean,27.902591,6.918356,8.324186,29.479390
std,7.547052,5.127311,6.754906,36.819928
min,16.000000,0.000000,1.000000,0.000000
25%,22.000000,3.000000,4.000000,5.000000
50%,28.000000,6.000000,6.000000,17.000000
75%,33.000000,9.000000,11.000000,40.000000
max,58.000000,74.000000,39.000000,404.000000


#### 1.2 - Conteo de nulos y duplicados

In [15]:
# Contar nulos por columnas
print(f"Nulos por Columna:\n {df.isnull().sum()}")

#Contar filas duplicadas exactas
print(f"\nFilas Duplicadas:\n {df.duplicated().sum()}")

Nulos por Columna:
 user_id                0
created_at             0
country                0
plan_type              0
user_age             754
post_id                0
post_category          0
post_created_at        0
votes_received         0
user_total_posts       0
days_since_signup      0
device_type            0
dtype: int64

Filas Duplicadas:
 172


##### 1.3 - Valores únicos y frecuencias

In [17]:
# Para cada columna categorica, ver que valores existen
print(df["plan_type"].value_counts())
print(f"\n {df["post_category"].value_counts()}")
print(f"\n {df["device_type"].value_counts()}")

plan_type
free            5978
pro             1460
enterprise       306
Free             208
 free            197
FREE             196
FreE             189
PRo               46
 pro              44
PRO               44
Pro               38
Pro               35
EnterPrise        13
ENTERPRISE        11
 enterprise        7
Enterprise         7
premium            1
vip                1
enterprise+        1
Name: count, dtype: int64

 post_category
tech           1187
life            913
sports          899
science         753
finance         739
gaming          727
music           614
health          601
education       592
travel          445
 tech            70
Tech             67
TECH             56
tehc             50
Life             46
Finance          44
 sport           42
sciense          42
gamming          40
 life            38
SPORTS           37
 finance         37
LIFE             36
Sports           36
 science         35
sporst           35
Gaming           35
Science  

##### 1.4 - Chequeos lógicos

In [21]:
# Convertir fechas temporalmente para el chequeo
signup = pd.to_datetime(df["created_at"], errors="coerce")
post = pd.to_datetime(df["post_created_at"], errors="coerce")

# Cuantos posts ocurren Antes del signup?
posts_before_signup = (post < signup).sum()
print(f"Posts antes del signup: {posts_before_signup}")

# Days_since_signup es consistente?
dias_calculados = (post - signup).dt.days
dias_originales = df["days_since_signup"]
inconsistentes = (dias_calculados != dias_originales).sum()
print(f"Days_since_signup inconsistentes: {inconsistentes}")

Posts antes del signup: 100
Days_since_signup inconsistentes: 4479


#### 2 - LIMPIEZA BÁSICA CON CRITERIO

##### PASO 2.1 - Eliminar duplicados exactos

In [22]:
duplicados_antes = df.duplicated().sum()
df = df.drop_duplicates()
duplicados_removidos = duplicados_antes

##### 2.2 - Normalización canónica (mapear variantes)

In [23]:
# Paso previo: poner todo en minusculas y quitar espacioss
df["plan_type"] = df["plan_type"].str.strip().str.lower()
df["device_type"] = df["device_type"].str.strip().str.lower()
df["post_category"] = df["post_category"].str.strip().str.lower()

# Definir diccionarios de mapeo para variantes conocidas
# (Los valores exactos dependeran de los que encuentres en 1.3)

mapeo_plan = {
    "free": "free",
    "pro": "pro",
    "enterprise": "enterprise"
}

mapeo_device = {
    "web": "web",
    "mobile": "mobile",
    "desktop": "desktop"
}

mapeo_category = {
    "sport": "sport",
    "tech": "tech",
    "life": "life",
    "gaming": "gaming",
    "music": "music",
    "education": "education",
    "health": "health",
    "science": "science",
    "travel": "travel",
    "finance": "finance"
}

df["plan_type"] = df["plan_type"].replace(mapeo_plan)
df["device_type"] = df["device_type"].replace(mapeo_device)
df["post_category"] = df["post_category"].replace(mapeo_category)


##### 2.3 - Convertir fechas a datetime

In [24]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["post_created_at"] = pd.to_datetime(df["post_created_at"], errors="coerce")

#Reportar fechas no parseables
fechas_nulas_signup = df["created_at"].isna().sum()
fechas_nulas_post = df["post_created_at"].isna().sum()
print(f"Fechas no parseables - signup: {fechas_nulas_signup}")
print(f"Fechas no parseables - post: {fechas_nulas_post}")

Fechas no parseables - signup: 1
Fechas no parseables - post: 1
